In [5]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from utils import classify_sentiment, normalize_text

df = pd.read_csv('../data/ai_apps_reviews_100000.csv')

if 'Sentiment' not in df.columns:
    df['Sentiment'] = df['score'].apply(classify_sentiment)

spacy_df = df.dropna(subset=['appName', 'content', 'score', 'Sentiment']).copy()
spacy_df['content'] = spacy_df['content'].apply(normalize_text)
spacy_df = spacy_df[spacy_df['content'] != ''].reset_index(drop=True)

print('Rows:', len(spacy_df))
print('Sentiment counts:', spacy_df['Sentiment'].value_counts().to_dict())

Rows: 233328
Sentiment counts: {'Positive': 194408, 'Negative': 28620, 'Neutral': 10300}


In [6]:
# Lemmatization + POS tagging (spaCy)

import spacy

nlp = spacy.load('en_core_web_sm')

sample_n = min(2000, len(spacy_df))
spacy_sample = spacy_df.sample(n=sample_n, random_state=42).copy()

def spacy_lemma_and_pos(text: str):
    doc = nlp(text)
    lemmas = [t.lemma_.lower() for t in doc if not t.is_space]
    pos = [t.pos_ for t in doc if not t.is_space]
    return ' '.join(lemmas), pos

lemmas = []
pos_tags = []
for txt in spacy_sample['content'].tolist():
    l, p = spacy_lemma_and_pos(txt)
    lemmas.append(l)
    pos_tags.append(p)

spacy_sample['lemma_content'] = lemmas
spacy_sample['pos_tags'] = pos_tags

def pos_dist(tag_lists):
    from collections import Counter
    c = Counter()
    for tags in tag_lists:
        c.update(tags)
    total = sum(c.values()) or 1
    return {k: v / total for k, v in c.most_common(10)}

print('Top POS tags by sentiment (sample):')
for sent in ['Negative', 'Neutral', 'Positive']:
    sub = spacy_sample[spacy_sample['Sentiment'] == sent]
    print(sent, 'n=', len(sub), pos_dist(sub['pos_tags'].tolist()))

Top POS tags by sentiment (sample):
Negative n= 251 {'NOUN': 0.1531489658377876, 'VERB': 0.14013478968161747, 'PRON': 0.11201487334417848, 'PUNCT': 0.08552172902626075, 'ADJ': 0.07971182895654194, 'AUX': 0.06785963281431559, 'ADV': 0.06576806878921683, 'DET': 0.06204973274459679, 'ADP': 0.06065535672786428, 'PROPN': 0.057866604694399255}
Neutral n= 61 {'NOUN': 0.15944272445820434, 'VERB': 0.12074303405572756, 'PRON': 0.11455108359133127, 'ADJ': 0.10371517027863777, 'ADP': 0.08204334365325078, 'AUX': 0.07894736842105263, 'PUNCT': 0.07430340557275542, 'ADV': 0.06811145510835913, 'DET': 0.06346749226006192, 'PART': 0.043343653250773995}
Positive n= 1688 {'NOUN': 0.1585825804753969, 'ADJ': 0.15560038593105868, 'PROPN': 0.10139461450749934, 'PRON': 0.09551793702306816, 'VERB': 0.09218489606174897, 'PUNCT': 0.0781510393825103, 'ADV': 0.07736163494430313, 'ADP': 0.05481975265327603, 'AUX': 0.048855363564599595, 'DET': 0.04657486185422331}
